# Realiza pruebas de inferencia 

Se utilizan varios videos de algunas Avenidas y Calles de Cochabamba

Elaborado por Alvaro Zambrana Sejas
Universidad Mayor de San Simón
2025

Configurar el entorno

## Pre-requisito

Instalar Anaconda3


```
conda create -n copiloto-virtual python=3.10
conda activate copiloto-virtual
```

In [9]:
# Verificamos la versió de Python
!python --version

Python 3.10.16


Los pasos para instalar las 3 versiones de YOLO son los siguientes:

0: Desinstalar cualquier versión previamente instalada, repetir este paso antes de instalar una versión diferente de YOLO

```
pip uninstall -y ultralytics
yolo -version
```

### YOLOv8

```
pip install ultralytics==8.2.103
yolo -version
```

### YOLOv10

Versión de yolo basado en Ultralytics 8.1.34

```
pip install -q git+https://github.com/THU-MIG/yolov10.git
yolo -version
```

### YOLO11

Versión específica con la que se realizó las inferencias en este notebook

```
pip install ultralytics==8.3.91
yolo -version
```

In [10]:
# Instalar las dependencias
!pip install opencv-python

In [12]:
from enum import Enum
from ultralytics import YOLO
import cv2
import datetime
import os

MODELS_PATH = '../../object-detection-models'

class DATASET_TYPES (Enum):
    ORIGINAL = 'ORIGINAL'
    AUMENTADO = 'AUMENTADO'    

YOLO_VERSIONS = ['YOLO11', 'YOLOv10', 'YOLOv8']

DATASET_PATHS={
    DATASET_TYPES.AUMENTADO.value: 'con_aumento_de_datos',
    DATASET_TYPES.ORIGINAL.value: 'sin_aumento_de_datos'
}

MODELS = {
    'ORIGINAL': {
        'YOLOv8': f"{MODELS_PATH}/{DATASET_PATHS[DATASET_TYPES.ORIGINAL.value]}/cbba_yolov8_model.pt",
        'YOLOv10': f'{MODELS_PATH}/{DATASET_PATHS[DATASET_TYPES.ORIGINAL.value]}/cbba_yolov10_model.pt',
        'YOLO11': f'{MODELS_PATH}/{DATASET_PATHS[DATASET_TYPES.ORIGINAL.value]}/cbba_yolo11_model.pt'
    },
    'AUMENTADO': {
        'YOLOv8': f'{MODELS_PATH}/{DATASET_PATHS[DATASET_TYPES.AUMENTADO.value]}/cbba_yolov8_model.pt',
        'YOLOv10': f'{MODELS_PATH}/{DATASET_PATHS[DATASET_TYPES.AUMENTADO.value]}/cbba_yolov10_model.pt',
        'YOLO11': f'{MODELS_PATH}/{DATASET_PATHS[DATASET_TYPES.AUMENTADO.value]}/cbba_yolo11_model.pt'
    }
}

VIDEOS_PATH = '../../demos'

VIDEOS = {
    'Av-Heroinas': f'{VIDEOS_PATH}/Av-Heroinas.mp4',
    'Av-America': f'{VIDEOS_PATH}/Av-America.mp4',
    'Av-Peru': f'{VIDEOS_PATH}/Av-Peru.mp4' 
}

"""
Infierencia sobre un video utilizando el modelo con la versión de YOLO utilizada en el entrenamiento
"""
def predecir_video(model, input_video_path, output_path, yolo_version):
    # Open the video file
    cap = cv2.VideoCapture(input_video_path)
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Define el codec y instancia VideoWriter para almacenar el video de salida con las anotaciones 
    # de los resultados de las predicciones e información de autoría
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 0.7 estricto por el tipo de app
        results = model(frame, conf=0.7, device=0)

        annotated_frame = results[0].plot()

        position = 10, height - 25
        add_author(annotated_frame, position, yolo_version, output_path)

        
        out.write(annotated_frame)
    
    cap.release()
    out.release()
    try:
        cv2.destroyAllWindows()
    except cv2.error as e:
        print(f"OpenCV error: {e}")

"""
Agrega información de autoría junto con la versión de YOLO utilizada en la inferencia
"""
def add_author(frame, position, yolo_version, output_path):
    x1, y1 = position
    font_size = 20
    offset=6*(font_size+3)
    add_label(frame, "Postulante: ALVARO ZAMBRANA SEJAS", (x1, y1 - offset + font_size*1))
    add_label(frame, "Tutor:  M.SC. ING. DANNY LUIS HUANCA SEVILLA", (x1, y1 - offset + font_size*2))
    add_label(frame, 
              "Diplomado: ESTADISTICA APLICADA A LA TOMA DE DECISIONES 3ra. VERSION - UMSS", 
              (x1, y1 - offset + font_size*3))
    add_label(frame, "Cochabamba - Bolivia", (x1, y1 - offset + font_size*4))
    add_label(frame, f'Fecha: {datetime.datetime.now().strftime("%Y-%m-%d")}', (x1, y1 - offset + font_size*5))

    file_name_with_extension = os.path.basename(output_path)

    add_label(frame, f'Ver. de YOLO utilizado: {yolo_version}; Archivo: {file_name_with_extension}', 
              (x1, y1 - offset + font_size*6))
    
def add_label(frame, label, position, text_color=(15, 217, 255)):
    cv2.putText(frame, label, position, cv2.FONT_HERSHEY_SIMPLEX, 2/3, text_color, 2)

In [13]:
import ultralytics

for model_version in DATASET_TYPES:
    for yolo_version in YOLO_VERSIONS:
     
        # uninstall ultralytics
        !pip uninstall -y ultralytics

        # instalar la versión correcta de YOLO para las tareas de inferencia
        if yolo_version == 'YOLOv8':
            !pip install ultralytics==8.2.103
        elif yolo_version == 'YOLOv10':
            !pip install -q git+https://github.com/THU-MIG/yolov10.git
        elif yolo_version == 'YOLO11':
            !pip install ultralytics
        
        !yolo -version
        ultralytics.checks()
        model_path = MODELS[model_version.value][yolo_version]
       
        model = YOLO(model_path)
        for video_name, video_path in VIDEOS.items():
            output_path = f'{VIDEOS_PATH}/{model_version.value}_{yolo_version}_{video_name}.mp4'
            print(f'Procesando video {video_path} con modelo {model_version.value} {yolo_version}')            
            predecir_video(model, video_path, output_path, yolo_version)
            print(f'Guardando video en {output_path}')

Ultralytics 8.3.91  Python-3.10.16 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
Setup complete  (12 CPUs, 31.9 GB RAM, 950.2/953.0 GB disk)
Procesando video ../../demos/Av-Heroinas.mp4 con modelo AUMENTADO YOLOv8

0: 736x1280 (no detections), 38.8ms
Speed: 11.6ms preprocess, 38.8ms inference, 0.7ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 (no detections), 16.5ms
Speed: 7.1ms preprocess, 16.5ms inference, 0.7ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 (no detections), 17.1ms
Speed: 4.4ms preprocess, 17.1ms inference, 1.0ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 (no detections), 10.6ms
Speed: 4.6ms preprocess, 10.6ms inference, 0.6ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 (no detections), 17.9ms
Speed: 4.2ms preprocess, 17.9ms inference, 1.0ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 (no detections), 9.2ms
Speed: 4.5ms preprocess, 9.2ms inference, 0.6ms postprocess